In [85]:
import requests
import dotenv
import os
import pprint

dotenv.load_dotenv()

True

In [86]:
auth_server_url = "https://sso.slf.ch/auth"
auth_realm = "SLF"
auth_client_id = "snowprofiler"
auth_username = "yannik.werner@slf.ch"
auth_password = os.getenv("SNOWPROFILER_PASSWORD")

snowprofiler_api_url = "https://snowprofiler-api.int.slf.ch"

In [87]:
def get_keycloak_token():
    r = requests.post(f'{auth_server_url}/realms/{auth_realm}/protocol/openid-connect/token',
                      headers={'content-type': 'application/x-www-form-urlencoded'},
                      data={'grant_type': 'password',
                            'client_id': auth_client_id,
                            'username': auth_username,
                            'password': auth_password})
    r.raise_for_status()
    return r.json().get('access_token')

In [88]:
def secure_profiles_search(end_date: str, multi_search_value: str | None = None, limit: int = 100, offset: int = 0, createdBy: str | None = None, state: str | None = None) -> list:
    token = get_keycloak_token()
    r = requests.get(f"{snowprofiler_api_url}/secure/profiles/search", params={"endDate": end_date, "multiSearchValue": multi_search_value, "limit": limit, "offset": offset, "createdBy": createdBy, "state": state}, headers={'accept': 'application/json', 'authorization': f'Bearer {token}'})
    r.raise_for_status()
    return r.json()

def secure_profiles_search_extended_detailed(start_date: str, end_date: str, maxSlopeAngle: int | None = None) -> dict:
    token = get_keycloak_token()
    r = requests.get(f"{snowprofiler_api_url}/secure/profiles/search/extended/detailed", params={"startDate": start_date, "endDate": end_date, "maxSlopeAngle": maxSlopeAngle}, headers={'accept': 'application/json', 'authorization': f'Bearer {token}'})
    r.raise_for_status()
    return r.json()

def export_json(profile_id: str) -> str:
    r = requests.get(f"{snowprofiler_api_url}/public/profiles/{profile_id}/")
    r.raise_for_status()
    return r.text

def export_caaml(profile_id: str) -> str:
    r = requests.get(f"{snowprofiler_api_url}/public/profiles/{profile_id}/export/caaml?timezone=Europe/Zurich&languageCode=en")
    r.raise_for_status()
    return r.text

def write_caaml_to_file(profile_id: str, filename: str):
    with open(filename, "w") as f:
        f.write(export_caaml(profile_id))

## TESTING

In [89]:
profiles = secure_profiles_search(end_date="2024-08-31T23:59:59Z")

In [90]:
print(len(profiles))
pprint.pprint(profiles[0])

100
{'aspect': None,
 'canEdit': False,
 'coordinates': [9.809176, 46.82946],
 'date': '2024-07-04T09:00:00Z',
 'elevation': 2536,
 'id': '76f2e9f3-e259-4ae6-bf6b-82ef5153233f',
 'includedProfiles': {'avalancheReleaseProfiles': 0,
                      'compressionTests': 0,
                      'densityProfiles': 1,
                      'extendedColumnTests': 0,
                      'ramResistanceProfiles': 0,
                      'rutschblockTests': 0,
                      'stratigraphyProfiles': 0,
                      'temperatureProfiles': 1},
 'isFlat': True,
 'locationName': 'Davos Weissfluhjoch / Versuchsfeld - 5WJ-R, SnowTinel '
                 'project',
 'observers': [{'name': 'Bavay'}],
 'slopeAngle': 0,
 'state': 'PUBLISHED'}


In [91]:
profiles = secure_profiles_search(end_date="2024-08-31T23:59:59Z", multi_search_value="Bavay")

In [92]:
print(len(profiles))

54


In [93]:
profile_headers = secure_profiles_search_extended_detailed("2023-09-01T00:00:00Z", "2024-08-31T23:59:59Z")

In [94]:
print(len(profile_headers))

1072


In [95]:
profiles = secure_profiles_search(end_date="2026-03-23T10:00:00Z")
print(len(profiles))
pprint.pprint(profiles[0])

100
{'aspect': 'NW',
 'canEdit': False,
 'coordinates': [8.038298, 46.24885],
 'date': '2026-03-22T14:20:00Z',
 'elevation': 2110,
 'id': '7ab4e661-56bd-4539-aebe-1733ec2b3c02',
 'includedProfiles': {'avalancheReleaseProfiles': 0,
                      'compressionTests': 0,
                      'densityProfiles': 0,
                      'extendedColumnTests': 1,
                      'ramResistanceProfiles': 1,
                      'rutschblockTests': 1,
                      'stratigraphyProfiles': 1,
                      'temperatureProfiles': 1},
 'isFlat': False,
 'locationName': 'Skilift Simplonpass',
 'observers': [{'name': 'Silvan Zenklusen'}],
 'slopeAngle': 34,
 'state': 'PUBLISHED'}


## ECT + RB Extraction

In [96]:
def extract_ect_rb(profiles: list[dict]) -> tuple[list[str], list[str]]:
    ect_ids = []
    rb_ids = []
    for profile in profiles:
        included = profile.get("includedProfiles")
        if included is None:
            continue
        if included.get("extendedColumnTests", 0) > 0:
            ect_ids.append(profile["id"])
        if included.get("rutschblockTests", 0) > 0:
            rb_ids.append(profile["id"])
    return ect_ids, rb_ids


In [97]:
ect_ids = []
rb_ids = []
counting_offset = 0
while True:
    new_profiles = secure_profiles_search(end_date="2026-03-23T10:00:00Z", offset=counting_offset)
    if len(new_profiles) == 0:
        break
    new_ect_ids, new_rb_ids = extract_ect_rb(new_profiles)
    ect_ids.extend(new_ect_ids)
    rb_ids.extend(new_rb_ids)
    counting_offset += 100
    if counting_offset % 1000 == 0:
        print(f"Processed {counting_offset} profiles")
        print(f"Last Timestamp: {new_profiles[-1]['date']}")
print(len(ect_ids))
print(len(rb_ids))

with open("ect_ids.txt", "w") as f:
    for id in ect_ids:
        f.write(f"{id}\n")

with open("rb_ids.txt", "w") as f:
    for id in rb_ids:
        f.write(f"{id}\n")


Processed 1000 profiles
Last Timestamp: 2025-03-31T11:00:01.957000Z
Processed 2000 profiles
Last Timestamp: 2024-04-23T14:30:00Z
Processed 3000 profiles
Last Timestamp: 2023-11-14T09:15:00Z
Processed 4000 profiles
Last Timestamp: 2022-04-21T12:30:00Z
Processed 5000 profiles
Last Timestamp: 2021-06-30T09:30:00Z
Processed 6000 profiles
Last Timestamp: 2020-12-30T08:45:00Z
Processed 7000 profiles
Last Timestamp: 2019-12-29T14:00:00Z
Processed 8000 profiles
Last Timestamp: 2019-01-15T17:00:00Z
Processed 9000 profiles
Last Timestamp: 2018-01-17T14:00:00Z
Processed 10000 profiles
Last Timestamp: 2017-01-26T15:00:00Z
Processed 11000 profiles
Last Timestamp: 2016-01-24T12:36:57Z
Processed 12000 profiles
Last Timestamp: 2015-01-27T12:08:00Z
Processed 13000 profiles
Last Timestamp: 2014-01-30T08:30:00Z
Processed 14000 profiles
Last Timestamp: 2013-02-15T13:00:00Z
Processed 15000 profiles
Last Timestamp: 2012-03-09T09:00:00Z
Processed 16000 profiles
Last Timestamp: 2011-02-25T13:25:00Z
Processed 

In [104]:
def read_ids_from_file(filename: str) -> list[str]:
    with open(filename, "r") as f:
        return [line.strip() for line in f.readlines()]

ids = read_ids_from_file("ect_ids.txt")

print(len(ids))

for i, id in enumerate(ids):
    if i % 100 == 0:
        print(f"Processed {i} profiles")
    write_caaml_to_file(id, f"ect/{id}.caaml.xml")

6535
Processed 0 profiles
Processed 100 profiles
Processed 200 profiles
Processed 300 profiles
Processed 400 profiles
Processed 500 profiles
Processed 600 profiles
Processed 700 profiles
Processed 800 profiles
Processed 900 profiles
Processed 1000 profiles
Processed 1100 profiles
Processed 1200 profiles
Processed 1300 profiles
Processed 1400 profiles
Processed 1500 profiles
Processed 1600 profiles
Processed 1700 profiles
Processed 1800 profiles
Processed 1900 profiles
Processed 2000 profiles
Processed 2100 profiles
Processed 2200 profiles
Processed 2300 profiles
Processed 2400 profiles
Processed 2500 profiles
Processed 2600 profiles
Processed 2700 profiles
Processed 2800 profiles
Processed 2900 profiles
Processed 3000 profiles
Processed 3100 profiles
Processed 3200 profiles
Processed 3300 profiles
Processed 3400 profiles
Processed 3500 profiles
Processed 3600 profiles
Processed 3700 profiles
Processed 3800 profiles
Processed 3900 profiles
Processed 4000 profiles
Processed 4100 profiles

In [106]:
def read_ids_from_file(filename: str) -> list[str]:
    with open(filename, "r") as f:
        return [line.strip() for line in f.readlines()]

ids = read_ids_from_file("rb_ids.txt")

print(len(ids))

for i, id in enumerate(ids):
    if i % 100 == 0:
        print(f"Processed {i} profiles")
    write_caaml_to_file(id, f"rb/{id}.caaml.xml")

13276
Processed 0 profiles
Processed 100 profiles
Processed 200 profiles
Processed 300 profiles
Processed 400 profiles
Processed 500 profiles
Processed 600 profiles
Processed 700 profiles
Processed 800 profiles
Processed 900 profiles
Processed 1000 profiles
Processed 1100 profiles
Processed 1200 profiles
Processed 1300 profiles
Processed 1400 profiles
Processed 1500 profiles
Processed 1600 profiles
Processed 1700 profiles
Processed 1800 profiles
Processed 1900 profiles
Processed 2000 profiles
Processed 2100 profiles
Processed 2200 profiles
Processed 2300 profiles
Processed 2400 profiles
Processed 2500 profiles
Processed 2600 profiles
Processed 2700 profiles
Processed 2800 profiles
Processed 2900 profiles
Processed 3000 profiles
Processed 3100 profiles
Processed 3200 profiles
Processed 3300 profiles
Processed 3400 profiles
Processed 3500 profiles
Processed 3600 profiles
Processed 3700 profiles
Processed 3800 profiles
Processed 3900 profiles
Processed 4000 profiles
Processed 4100 profile

## Result ANALYSIS

In [ ]:
ect_folder = "../data/raw/slf-ect"
rb_folder = "../data/raw/slf-rb"

for i, file in enumerate(os.listdir(ect_folder)):
    if i % 100 == 0:
        print(f"Processed {i} files")
    with open(os.path.join(ect_folder, file), "r") as f:
        data = f.read()
    if "Rutschblock" in data:
        print(file)